# Memory-Augmented Agent Pattern
### From stateless to stateful intelligence

A stateless agent treats every interaction as new — no history, no context, no learning. A memory-augmented agent stores and retrieves short-term and long-term memory, maintains context across sessions, and adapts from historical data and outcomes.

| | Stateless Agent | Memory-Augmented Agent |
|---|---|---|
| **Context** | Every interaction starts from zero | Maintains context across sessions |
| **Personalization** | Generic responses | Adapts from past decisions and preferences |
| **Continuity** | Cannot follow up on prior work | Picks up where it left off |

This pattern uses **Amazon Bedrock AgentCore Memory** — managed short-term memory (session history) and long-term memory (extracted facts, summaries, preferences) that persist across sessions.

### Business Use Case: AnyComp Telecom Multi-Session Customer Support

A customer contacts AnyComp Telecom about a billing overcharge. The agent resolves it. Two days later, the same customer calls back about a related issue. With memory, the agent recalls the previous interaction — the overcharge, the resolution, the customer’s communication preference — and provides personalized, context-aware support without asking the customer to repeat themselves.

### Architecture

```
Session 1                              AgentCore Memory
  User → Agent → tools → response  ──→  STM: session events
                                        LTM: facts, preferences, summaries

         [time passes...]

Session 2                              AgentCore Memory
  User → Agent ───────────────────←  retrieve_memories()
           │                            │
           └── context from Session 1 ─┘
           │
    Personalized, context-aware response
```

### What You’ll See

1. **Before (stateless)**: Two sessions with no memory — the agent has no idea about the first call
2. **After (AgentCore Memory)**: Same two sessions — the agent recalls the first interaction and provides continuity

## Install Dependencies

In [ ]:
!pip install -q strands-agents bedrock-agentcore boto3 typing_extensions>=4.12.0

**Restart the kernel** after installing, then continue.

## Setup

In [ ]:
import boto3
import json
import time
import uuid
from datetime import datetime

from strands import Agent, tool
from strands.models import BedrockModel

REGION = 'us-east-1'

# Simple customer lookup tool
CUSTOMERS = {
    "CUST-001": {
        "name": "Maria Chen",
        "plan": "Premium",
        "tenure_years": 5,
        "monthly_bill": 95.00,
        "email": "maria.chen@example.com",
        "preference": "email",
    }
}

print('Setup complete')

## Customer Support Tools

In [ ]:
@tool
def lookup_customer(customer_id: str) -> str:
    """Look up customer account details.
    Args:
        customer_id: Customer ID (e.g. CUST-001)
    """
    customer = CUSTOMERS.get(customer_id)
    return json.dumps(customer) if customer else f"Customer {customer_id} not found."

@tool
def check_billing_history(customer_id: str) -> str:
    """Check recent billing history for a customer.
    Args:
        customer_id: Customer ID
    """
    # Mock billing data
    return json.dumps({
        "customer_id": customer_id,
        "recent_charges": [
            {"date": "2026-03-01", "amount": 95.00, "description": "Monthly Premium plan"},
            {"date": "2026-03-05", "amount": 45.00, "description": "International roaming - ERROR: duplicate charge"},
            {"date": "2026-03-05", "amount": 45.00, "description": "International roaming"},
        ],
        "balance": 185.00,
    })

@tool
def issue_refund(customer_id: str, amount: float, reason: str) -> str:
    """Issue a refund to a customer account.
    Args:
        customer_id: Customer ID
        amount: Refund amount in dollars
        reason: Reason for the refund
    """
    return json.dumps({
        "status": "refund_issued",
        "customer_id": customer_id,
        "amount": amount,
        "reason": reason,
        "refund_id": f"REF-{uuid.uuid4().hex[:8].upper()}",
    })


support_tools = [lookup_customer, check_billing_history, issue_refund]

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

print("Tools defined: lookup_customer, check_billing_history, issue_refund")

## Before: Stateless Agent (No Memory)

Two support sessions with the same customer. The agent has no memory — each session starts from zero.

In [ ]:
SYSTEM_PROMPT = (
    "You are a customer support agent for AnyComp Telecom. "
    "Use your tools to look up accounts and resolve billing issues. "
    "Be helpful, concise, and reference specific account details."
)

# Session 1: Customer reports a billing issue
print("=" * 60)
print("SESSION 1 (stateless): Billing overcharge")
print("=" * 60)
agent_s1 = Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=support_tools)
result_s1 = agent_s1(
    "Hi, I'm customer CUST-001. I was charged twice for international roaming on March 5th. "
    "Can you check my billing and fix this? I prefer to be contacted by email."
)
print(result_s1)

# Session 2: Same customer follows up (new agent instance = no memory)
print()
print("=" * 60)
print("SESSION 2 (stateless): Follow-up — no memory of Session 1")
print("=" * 60)
agent_s2 = Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=support_tools)
result_s2 = agent_s2(
    "Hi, I'm customer CUST-001. I called a couple days ago about a billing issue. "
    "Has my refund been processed? Also, can you make sure I don't get double-charged again?"
)
print(result_s2)
print()
print("❌ The agent has no idea about Session 1 — it cannot reference the refund or the customer's email preference.")

## Create AgentCore Memory

We create a memory resource with both short-term memory (session history) and long-term memory (extracted facts and summaries that persist across sessions).

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

memory_client = MemoryClient(region_name=REGION)
MEMORY_NAME = 'AnyComp_CustomerSupport_Memory'

try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Customer support memory with session summaries and preference extraction",
        strategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "SessionSummarizer",
                    "namespaceTemplates": ["/summaries/{actorId}/{sessionId}/"]
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "PreferenceLearner",
                    "namespaceTemplates": ["/preferences/{actorId}/"]
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "FactExtractor",
                    "namespaceTemplates": ["/facts/{actorId}/"]
                }
            },
        ]
    )
    MEMORY_ID = memory.get("id")
    print(f"✅ Created AgentCore Memory: {MEMORY_NAME}")
except Exception as e:
    if "already exists" in str(e):
        MEMORY_ID = ''
        for m in memory_client.list_memories():
            if m.get("name") == MEMORY_NAME:
                MEMORY_ID = m.get("id")
                break
        print(f"ℹ️  Memory already exists")
    else:
        raise

print(f"   Memory ID: {MEMORY_ID}")

## After: Strands Agent with AgentCore Memory

Same two sessions, but now the agent has AgentCore Memory. Session 1’s conversation is captured. Session 2 retrieves the context and picks up where it left off.

In [ ]:
ACTOR_ID = 'CUST-001'
SESSION_1_ID = f'session-1-{uuid.uuid4().hex[:8]}'

# Session 1 with memory
memory_config_s1 = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id=SESSION_1_ID,
    actor_id=ACTOR_ID,
)
session_mgr_s1 = AgentCoreMemorySessionManager(memory_config_s1, REGION)

agent_m1 = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=support_tools,
    session_manager=session_mgr_s1,
)

print("=" * 60)
print("SESSION 1 (with memory): Billing overcharge")
print("=" * 60)
result_m1 = agent_m1(
    "Hi, I'm customer CUST-001. I was charged twice for international roaming on March 5th. "
    "Can you check my billing and fix this? I prefer to be contacted by email."
)
print(result_m1)

# Wait for LTM extraction
print("\n⏳ Waiting 30s for long-term memory extraction...")
time.sleep(30)

# Session 2 with memory — new session, same actor
SESSION_2_ID = f'session-2-{uuid.uuid4().hex[:8]}'

memory_config_s2 = AgentCoreMemoryConfig(
    memory_id=MEMORY_ID,
    session_id=SESSION_2_ID,
    actor_id=ACTOR_ID,
    retrieval_config={
        f"/summaries/{ACTOR_ID}/{SESSION_1_ID}/": RetrievalConfig(top_k=3, relevance_score=0.5),
        f"/preferences/{ACTOR_ID}/": RetrievalConfig(top_k=3, relevance_score=0.5),
        f"/facts/{ACTOR_ID}/": RetrievalConfig(top_k=5, relevance_score=0.5),
    },
)
session_mgr_s2 = AgentCoreMemorySessionManager(memory_config_s2, REGION)

agent_m2 = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT + (
        " You have access to memory from previous sessions. "
        "Use it to provide continuity — reference prior interactions, "
        "known preferences, and past resolutions."
    ),
    tools=support_tools,
    session_manager=session_mgr_s2,
)

print()
print("=" * 60)
print("SESSION 2 (with memory): Follow-up — agent recalls Session 1")
print("=" * 60)
result_m2 = agent_m2(
    "Hi, I'm customer CUST-001. I called a couple days ago about a billing issue. "
    "Has my refund been processed? Also, can you make sure I don't get double-charged again?"
)
print(result_m2)
print()
print("✅ The agent now recalls Session 1 — the refund, the duplicate charge, and the email preference.")

## Cleanup (Optional)

In [ ]:
try:
    memory_client.delete_memory(memory_id=MEMORY_ID)
    print(f"✅ Deleted AgentCore Memory: {MEMORY_ID}")
except Exception as e:
    print(f"⚠️  {e}")

## Key Takeaway

Memory transforms disconnected transactions into continuous workflows. AgentCore Memory provides the managed infrastructure — short-term memory captures session history, long-term memory extracts facts and preferences that persist across sessions. The agent doesn’t need to be re-architected — you add a `session_manager` (Strands) or inject retrieved context into the prompt (LangChain, CrewAI).